In [8]:
# VERSION RAPIDA DE EJECUCION DEL DINEOF

using TSVD
using LinearAlgebra
using Arpack
using Statistics
using Optim
using NCDatasets
using GeoArrays
using Missings
using ArchGDAL
push!(LOAD_PATH, raw"D:\DATA_DINEOF\DINEOF-jmbeckers\DINEOF.jl-master\src", ".") 
using DINEOF

#-----Funcion para exportar a geotiff---------------------------
function exportar_geotiff_apilado(file_apilado::String, file_output::String, A_stack::Array{Float32, 3})
    """
    exportar_geotiff_apilado(file_apilado, file_output, A_stack)

    Guarda un apilado 3D (`A_stack`, en formato filas × columnas × bandas) como 
    un GeoTIFF multibanda, usando la proyección y georreferenciación de un 
    raster de referencia (`file_apilado`).

    # Args
    - `file_apilado::String`: Ruta del GeoTIFF de referencia (para CRS y GeoTransform).
    - `file_output::String`: Ruta de salida del GeoTIFF.
    - `A_stack::Array{Float32,3}`: Apilado a exportar.

    Cada capa del apilado se escribe como una banda independiente en el GeoTIFF.
   """
    dataset = ArchGDAL.readraster(file_apilado)
    gt  = ArchGDAL.getgeotransform(dataset)
    crs = ArchGDAL.getproj(dataset)
    ncols, nrows, nbands = size(A_stack)
    raster_salida = ArchGDAL.create(file_output,
        driver = ArchGDAL.getdriver("GTiff"),
        width  = ncols,
        height = nrows,
        nbands = nbands,
        dtype  = ArchGDAL.pixeltype(ArchGDAL.getband(dataset, 1)),
        options = ["BIGTIFF=YES", "COMPRESS=compressmethod"] # EL archivo pesa mas
        #options = ["BIGTIFF=YES", "COMPRESS=DEFLATE"] # para mayor compresion usar "COMPRESS=DEFLATE"
    )
    ArchGDAL.setgeotransform!(raster_salida, gt)
    ArchGDAL.setproj!(raster_salida, crs)
    for b in 1:nbands
        ArchGDAL.write!(raster_salida, A_stack[:, :, b], b)
    end
    ArchGDAL.destroy(raster_salida)
    println("✅ Exportado: $file_output ($nrows filas x $ncols columnas x $nbands bandas)")
end

#-------------------------------------------------------------------------------------------------------

function procesar_dineof(file_apilado::String)
    # Leyendo la data
    dataset = ArchGDAL.readraster(file_apilado)
    nbandas, ncols, nfilas = ArchGDAL.nraster(dataset), ArchGDAL.width(dataset), ArchGDAL.height(dataset)
    array_data = dataset[:, :, 1:nbandas]
    # Procesando con DINEOF
    array_dineof = copy(array_data)
    @time XA, offset, U, S, V, cvEOF, cvarray, errmap, musquare = DINEOFrun(array_dineof, [1, 1, 2]; eofmax=20,minimumcoverage=(0.0,0.0)) #minimumcoverage=(0.0,0.0) es para que interpole las bandas vacias
    X = copy(array_dineof)
    DINEOF_fuse!(X, XA, 0)  # solo rellena las areas sin datos con las data interpolada
     #----Exportando---------------------------------------------------------------------------------------------------------------------
    nombre_con_extension = basename(file_apilado) # Obtener el nombre del archivo con extensión
    nombre_sin_extension = splitext(nombre_con_extension)[1]# Separar el nombre de la extensión
    localdir = pwd()   # devuelve el directorio donde se ejecuta Julia
    outdir = joinpath(localdir,"Resultados_dineof_rapido") #
    mkpath(outdir)
    name_interpolado=nombre_sin_extension * "_INTERPOLADO.tif"
    name_errmap = nombre_sin_extension * "_ERRORMAP.tif"
    file_output_interpolado=joinpath(outdir,name_interpolado)
    file_output_errmap=joinpath(outdir,name_errmap)
    println("===========================================================================================")
    exportar_geotiff_apilado(file_apilado, file_output_interpolado, Float32.(X))#campo reconstruido
    exportar_geotiff_apilado(file_apilado, file_output_errmap, Float32.(errmap)) #mapa de incertidumbre por píxel.
    println("===========================================================================================")
    println("FIN DE PROCESO")
end

# Ejemplo de uso:
# Solo se requiere el archivo de entrada. Los resultados se guardan en la carpeta "Resultados_dineof_monse".
file_apilado = raw"D:\DATA_DINEOF\DATA\Tile_12.tif"
procesar_dineof(file_apilado)


Raw data variance and mean: 0.00207084 and 0.005244507
Number of missing points (including possible masks): 1767584 out of 156216060
Number of data points before elimination of low coverage regions is 154448476 and cv fraction 0.010816461536337852
Number of data points after elimination of low coverage regions is 154448476 and cv fraction 0.010816461536337852
(mean(X2D), meanmatrix, meanmiss, datamean) = (7.6966714f-8, -3.5408249f-10, 6.792864f-6, 0.005244507f0)
svds!: variance and mean of the entry matrix: 0.0020708449 , 7.6966714e-8 ; intial variance at points to fill in: 0.00206898 
Eof loop 1 with rms cv misfit: 0.00634346318589227 
Eof loop 2 with rms cv misfit: 0.005706927675898052 
Eof loop 3 with rms cv misfit: 0.005434753421366494 
Eof loop 4 with rms cv misfit: 0.005250744827590312 
Eof loop 5 with rms cv misfit: 0.005117001353112164 
Eof loop 6 with rms cv misfit: 0.00504218173465445 
Eof loop 7 with rms cv misfit: 0.004958139661425084 
Eof loop 8 with rms cv misfit: 0.00492

┌ Warning: Initial Variance has been increased for filtered matrix  by factor 1.368869
└ @ DINEOF D:\DATA_DINEOF\DINEOF-jmbeckers\DINEOF.jl-master\src\DINEOF_svds!.jl:320


tutu = Results of Optimization Algorithm
 * Status: success
 * Algorithm: Brent's Method
 * Search Interval: [0.000007, 0.001411]
 * Minimizer: 1.394723e-03
 * Minimum: 2.737919e+05
 * Iterations: 8
 * Convergence: max(|x - x_upper|, |x - x_lower|) <= 2*(1.0e-02*|x|+1.0e-07): true
 * Objective Function Calls: 9
CV estimator from EOF 2.423415168868238e-5 is now 1.4584905936964589e-5 if OI is used
Optimal musquare is 0.001394722646053271
Relative error on reconstruction 5.724053272285436e-5, relative error on CV estimator 0.1585371661279047
The two criteria to compare OI and EOF are: reconstruction 8941.89049426538, closest CV 264849.97021295177
Estimated musquare 1.4106927e-5 was inflated by factor 98.8679299335416 into 0.001394722646053271
This optimal value provides OI interpolation CV estimator 1.4584905936964589e-5
Final Error Map with mean error variance of reconstruction: 3.843330726526217e-6 
musquare = 0.001394722646053271
957.617592 seconds (1.28 G allocations: 740.087 GiB, 7.5